<a href="https://colab.research.google.com/github/edusatyaki/ProjectDrive/blob/main/Group-C/Olist%20E-Commerce%20Analytics/Data%20Cleaning/Brazil_Ecommerce_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Brazil E-Commerce Data Analysis

### 1. Import Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

plt.rcParams['figure.figsize'] = (10,6)

###Step 2. Load Dataset

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
DATA_PATH = '/content/drive/MyDrive/Olist_Dataset/'

# 2. Load the datasets
datasets = {
    'customers': pd.read_csv(DATA_PATH + 'olist_customers_dataset.csv'),
    'geolocation': pd.read_csv(DATA_PATH + 'olist_geolocation_dataset.csv'),
    'order_items': pd.read_csv(DATA_PATH + 'olist_order_items_dataset.csv'),
    'order_payments': pd.read_csv(DATA_PATH + 'olist_order_payments_dataset.csv'),
    'order_reviews': pd.read_csv(DATA_PATH + 'olist_order_reviews_dataset.csv'),
    'orders': pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv'),
    'products': pd.read_csv(DATA_PATH + 'olist_products_dataset.csv'),
    'sellers': pd.read_csv(DATA_PATH + 'olist_sellers_dataset.csv'),
    'translation': pd.read_csv(DATA_PATH + 'product_category_name_translation.csv')
}

# 3. Verify they loaded correctly
print("=== RAW DATASET DOCUMENTATION ===")
for name, df in datasets.items():
    print(f"Dataset: {name:<15} | Rows: {df.shape[0]:<7} | Columns: {df.shape[1]}")

=== RAW DATASET DOCUMENTATION ===
Dataset: customers       | Rows: 99441   | Columns: 5
Dataset: geolocation     | Rows: 1000163 | Columns: 5
Dataset: order_items     | Rows: 112650  | Columns: 7
Dataset: order_payments  | Rows: 103886  | Columns: 5
Dataset: order_reviews   | Rows: 99224   | Columns: 7
Dataset: orders          | Rows: 99441   | Columns: 8
Dataset: products        | Rows: 32951   | Columns: 9
Dataset: sellers         | Rows: 3095    | Columns: 4
Dataset: translation     | Rows: 71      | Columns: 2


### Step 3: Handle Duplicates & Missing Values

In this step, we are cleaning the datasets to ensure our analysis is accurate. Specifically, this code will:
* **Identify Duplicates:** Scan all datasets and print the number of exact duplicate rows.
* **Fix Missing Reviews:** Fill empty review titles with `"No Title"` and empty messages with `"No Comment"`.
* **Impute Product Data:** Replace missing physical product measurements (weight, dimensions, etc.) with the median value of those columns.
* **Deduplicate Geolocation:** Group the geolocation table by zip code to calculate a single, average latitude and longitude per zip code, removing redundant coordinates.

In [6]:
# 1. Check duplicate counts across all datasets
print("=== DUPLICATE SUMMARY ===")
for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} exact duplicate rows")

# 2. Clean 'order_reviews' missing text fields
datasets['order_reviews']['review_comment_title'] = datasets['order_reviews']['review_comment_title'].fillna('No Title')
datasets['order_reviews']['review_comment_message'] = datasets['order_reviews']['review_comment_message'].fillna('No Comment')

# 3. Clean 'products' missing physical properties with median values
num_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_photos_qty']
for col in num_cols:
    datasets['products'][col] = datasets['products'][col].fillna(datasets['products'][col].median())

# 4. Deduplicate Geolocation table (Zip codes appear multiple times)
datasets['geolocation_clean'] = datasets['geolocation'].groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean',
    'geolocation_city': 'first',
    'geolocation_state': 'first'
}).reset_index()

print("\nMissing values and duplicates successfully handled!")

=== DUPLICATE SUMMARY ===
customers: 0 exact duplicate rows
geolocation: 261831 exact duplicate rows
order_items: 0 exact duplicate rows
order_payments: 0 exact duplicate rows
order_reviews: 0 exact duplicate rows
orders: 0 exact duplicate rows
products: 0 exact duplicate rows
sellers: 0 exact duplicate rows
translation: 0 exact duplicate rows

Missing values and duplicates successfully handled!


### Step 4: Correct Data Types & Translate Categories

* **Datetime Conversion:** Converted all timestamp strings in the `orders`, `order_items`, and `order_reviews` tables into native pandas `datetime` objects.
* **Translation:** Merged the `product_category_name_translation` table into the `products` table to provide English category names, filling any missing translations with `"unknown"`.

In [7]:
# 1. Convert timestamp strings to datetime objects
date_columns = {
    'orders': ['order_purchase_timestamp', 'order_approved_at',
               'order_delivered_carrier_date', 'order_delivered_customer_date',
               'order_estimated_delivery_date'],
    'order_items': ['shipping_limit_date'],
    'order_reviews': ['review_creation_date', 'review_answer_timestamp']
}

for table, cols in date_columns.items():
    for col in cols:
        datasets[table][col] = pd.to_datetime(datasets[table][col])

# 2. Merge English category translations into products dataframe
datasets['products'] = datasets['products'].merge(
    datasets['translation'],
    on='product_category_name',
    how='left'
)

# Fill any categories that didn't have a translation with 'unknown'
datasets['products']['product_category_name_english'] = datasets['products']['product_category_name_english'].fillna('unknown')

# 3. Validate the changes
print("=== DATA TYPES VALIDATION ===")
print("Data types for Orders table dates:\n", datasets['orders'].dtypes[date_columns['orders']])
print("\nData types corrected and categories translated successfully!")

=== DATA TYPES VALIDATION ===
Data types for Orders table dates:
 order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

Data types corrected and categories translated successfully!


### Step 5: Merge & Export Master Dataset

* **Consolidation:** Joined the core relational datasets (`orders`, `order_items`, `products`, `customers`, and `order_payments`) into a single master analytical dataframe.
* **Export:** Saved the resulting `master_df` and the deduplicated `geolocation_clean` table as CSV files back to Google Drive for future Exploratory Data Analysis (EDA) and dashboarding.

In [8]:
# 1. Merge datasets into a primary analytical table
master_df = datasets['orders'].merge(datasets['order_items'], on='order_id', how='inner') \
                              .merge(datasets['products'], on='product_id', how='left') \
                              .merge(datasets['customers'], on='customer_id', how='left') \
                              .merge(datasets['order_payments'], on='order_id', how='left')

print("Master Cleaned Shape:", master_df.shape)

# 2. Save clean datasets to Google Drive
print("Exporting to Google Drive... (this might take a minute)")
master_df.to_csv(DATA_PATH + 'olist_master_cleaned.csv', index=False)
datasets['geolocation_clean'].to_csv(DATA_PATH + 'olist_geolocation_cleaned.csv', index=False)

print(" Clean datasets exported to Google Drive successfully!")

Master Cleaned Shape: (117604, 31)
Exporting to Google Drive... (this might take a minute)
 Clean datasets exported to Google Drive successfully!
